In [1]:
import os
import time
import requests
import pandas as pd


def fetch_coin_markets(vs_currency="usd", per_page=250, page=1):
    url = "https://api.coingecko.com/api/v3/coins/markets"

    params = {
        "vs_currency": vs_currency,
        "order": "market_cap_desc",
        "per_page": per_page,
        "page": page,
        "sparkline": "false",
        "price_change_percentage": "24h,7d,30d"
    }

    headers = {
        "accept": "application/json",
        "User-Agent": "stat386-project/1.0"
    }

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()


def build_dataset():
    all_data = []

    for page in [1, 2]:
        page_data = fetch_coin_markets(per_page=250, page=page)
        all_data.extend(page_data)
        time.sleep(1)

    df = pd.DataFrame(all_data)

    keep_cols = [
        "id",
        "symbol",
        "name",
        "current_price",
        "market_cap",
        "market_cap_rank",
        "total_volume",
        "high_24h",
        "low_24h",
        "price_change_percentage_24h",
        "circulating_supply",
        "last_updated"
    ]

    df = df[keep_cols].copy()

    df = df.rename(columns={
        "id": "coin_id",
        "symbol": "ticker",
        "name": "coin_name",
        "current_price": "price_usd",
        "market_cap": "market_cap_usd",
        "market_cap_rank": "market_cap_rank",
        "total_volume": "volume_usd_24h",
        "high_24h": "high_usd_24h",
        "low_24h": "low_usd_24h",
        "price_change_percentage_24h": "pct_change_24h",
        "circulating_supply": "circulating_supply",
        "last_updated": "last_updated_utc"
    })

    df["ticker"] = df["ticker"].str.upper()
    df["last_updated_utc"] = pd.to_datetime(df["last_updated_utc"], errors="coerce")
    df = df.drop_duplicates(subset=["coin_id"]).reset_index(drop=True)

    return df


def main():
    df = build_dataset()

    os.makedirs("data", exist_ok=True)
    output_path = "data/crypto_markets.csv"
    df.to_csv(output_path, index=False)

    print(f"Saved dataset to: {output_path}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print(df.head())


if __name__ == "__main__":
    main()

Saved dataset to: data/crypto_markets.csv
Rows: 500
Columns: 12
       coin_id ticker coin_name  price_usd  market_cap_usd  market_cap_rank  \
0      bitcoin    BTC   Bitcoin   69408.00   1388236061134                1   
1     ethereum    ETH  Ethereum    2024.53    244344564022                2   
2       tether   USDT    Tether       1.00    183960319199                3   
3  binancecoin    BNB       BNB     643.66     87767867799                4   
4       ripple    XRP       XRP       1.37     83768015168                5   

   volume_usd_24h  high_usd_24h   low_usd_24h  pct_change_24h  \
0    4.537870e+10      71230.00  69034.000000        -0.16941   
1    1.884567e+10       2082.55   2010.470000         0.05093   
2    7.049866e+10          1.00      0.999893         0.00204   
3    9.921448e+08        655.24    636.750000         0.67164   
4    2.323530e+09          1.41      1.370000        -0.81602   

   circulating_supply                 last_updated_utc  
0        2.00